## Testdaten
Für das Projekt müssen zumindest einmalig Testdaten generiert werden, die zum RAG-Datensatz passen. Das werde ich wieder mit einem LLM erstellen lassen, ich denke, die Aufgabe ist nicht schwer wenn es nur dedizierte Daten erhält. Die Testmenge wird überschaubar bleiben und kann in handarbeit evaluiert werden.

In [13]:
import os
import json
import pandas as pd
from mistralai import Mistral
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv('MISTRAL_API_KEY')
model = 'mistral-medium-2508'
client = Mistral(api_key=api_key, timeout_ms=120000)

In [14]:
# Requestfunktion
def agent_request(system_promt, schema, content):

    response = client.chat.complete(
        model = model,
        messages = [
            {
                'role': 'system',
                'content': system_promt
            },
            {
                'role': 'user',
                'content': content,
            }
        ],
        response_format = {
            "type": "json_object",
            "json_schema": schema
        }
    )

    return response

In [15]:
# Daten laden
df = pd.read_json('../data/processed/products_chunked.jsonl', lines=True)

## Generation

Es soll zwei Datensätze an Testdaten geben. Zum eine Fragen, für deren Beantwortung ein bestimmter Chunk gefunden werden muss, zum anderen solche, deren Antworten über mehrere Chunks verteilt ist.

Für die Singe-Chunk-Fragen werden nur technische Daten verwendet. Um diese filtern zu können muss auf die Metadaten zugegriffen werden, die ein JSON-Objekt mit den Daten enthalten. Eine Möglichkeit ist, den Typen in eine eigene Spalte zu schreiben, ist letztlich ja pro Chunk typisch.

In [16]:
df = pd.read_json('../data/processed/products_chunked.jsonl', lines=True)
df['chunk_type'] = df['metadata'].apply(pd.Series)['chunk_type']

### Single Chunk Questions

Aus den gefilterten Datensätzen sollen zufällig Datensätze gezogen und an das LLM zur Generierung der Testfragen gesendet werden.

In [17]:
with open('../data/promts/test_single_chunk_agent.md', 'r') as f:
    specs_prompt = f.read()

with open('../data/promts/test_single_chunk_schema.json', 'r') as f:
    specs_schema = json.load(f)

specs_df = df[df['chunk_type'] == 'spec']

In [24]:
specs_quests = []
specs_samples = specs_df.sample(n=25)

print(specs_samples)
for index, spec in specs_samples.iterrows():

    response = agent_request(specs_prompt, specs_schema, spec['document'])

    print("=" * 50)
    print("LLM Response:")
    print(response)
    print("=" * 50)
    
    print(response.choices[0].message.content)
    for question in json.loads(response.choices[0].message.content):
        specs_quests.append({
            'product_id': spec['metadata']['product_id'],
            'question': question,
            'answer': spec['document'],
            'chunk_id': spec['id']
        })
    
with open('../data/tests/specs_question.json', 'w', encoding='utf-8') as f:
    json.dump(specs_quests, f, ensure_ascii=False, indent=2)

# print(specs_quests)

                                                     id  \
2765  SFTvh-1501-Perfection-Laborgefriergeraet-mit-U...   
1175  Kirsch-FROSTER-LABEX-730-ULTIMATE-Laborgefrier...   
2006  LCexv-4010-MediLine-30-bis-160-90-bis-300C_spe...   
305   Kirsch-LABEX-288-PRO-ACTIVE-Laborkuehlschrank_...   
2685  SFFsg-5501-Performance-mit-Schubfaechern-90C-b...   
1747  Kirsch-LABEX-468-PRO-ACTIVE-Laborkuehlschrank_...   
4116  Kirsch-FROSTER-LABEX-530-PRO-ACTIVE-Laborgefri...   
5126  Kirsch-FROSTER-LABEX-71-ESSENTIAL-Laborkuehlsc...   
1269  Kirsch-FROSTER-LABO-330-ULTIMATE-Laborgefriers...   
4055  Kirsch-MED-468-ULTIMATE-Medikamentenkuehlschra...   
2331       SFFvh-5501-Perfection-90-C-bis-350-C_spec_11   
2087  Kirsch-MED-340-PRO-ACTIVE-Medikamentenkuehlsch...   
167   HMFvh-5511-H63-Perfection-50-C-mit-Glastuer_sp...   
4932  SFPvh-6501-Perfection-Laborgefriergeraet-mit-U...   
3283  SRPvh-6501-Perfection-Laborkuehlgeraet-mit-Uml...   
1696  Kirsch-FROSTER-MED-95-PRO-ACTIVE-Medikamentenk... 

JSONDecodeError: Extra data: line 6 column 1 (char 257)

### Multi Chunk Questions

In [ ]:
with open('../data/promts/test_multi_chunk_agent.md', 'r') as f:
    specs_prompt = f.read()

with open('../data/promts/test_multi_chunk_schema.json', 'r') as f:
    descs_schema = json.load(f)

descs_df = df[df['chunk_type'] == 'desc']

In [ ]:
descs_samples = descs_df.sample(n=3)

for index, row in descs_samples.iterrows():
    print(f"{row['document']}")

    response = agent_request(specs_prompt, descs_schema, row['document'])

with open('../data/tests/multi_question.json', 'w', encoding='utf-8') as f:
    json.dump(specs_quests, f, ensure_ascii=False, indent=2)